# Notebook 04 — Re-evaluation on Real Logs (Way 3 Inference)

Evaluate NB03 models on `labeled_access.log` using Way 3 inference:
extract only query parameter values (skip path-only URLs).

**NB02 RF baseline to beat:** FP=5,666  FP/10k=81.1  Recall=0.75


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import joblib, re, os, time, json, urllib.parse, warnings
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

for d in ['results/metrics', 'results/figures', 'results/attack_logs']:
    os.makedirs(d, exist_ok=True)

SYMBOLS = ["'",'"',";","--","#","/*","*/","*","+","|","(",")",">","<","\\","/","="]

def extract_symbol_frequencies(text):
    c = Counter()
    for sym in SYMBOLS:
        c[sym] = str(text).count(sym)
    return c

def build_symbol_matrix(queries):
    rows = [extract_symbol_frequencies(q) for q in queries]
    return csr_matrix(pd.DataFrame(rows).fillna(0).values)

def extract_query_values(url):
    """Return joined query param VALUES, or None if no query string."""
    parsed = urllib.parse.urlparse(url)
    params = urllib.parse.parse_qs(parsed.query, keep_blank_values=False)
    values = [urllib.parse.unquote(v).strip()
              for vlist in params.values() for v in vlist
              if urllib.parse.unquote(v).strip()]
    return ' '.join(values) if values else None

METHOD_RE = re.compile(r'"(GET|POST|HEAD|PUT|DELETE|OPTIONS|PATCH|TRACE|CONNECT)\s+([^"]+)\s+HTTP')

print('Setup complete.')


Setup complete.


## 2. Load Log & Apply Way 3 Inference

In [2]:
records = []
with open('../logs/labeled_access.log', 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.strip().split(',', 2)
        if len(parts) == 3:
            records.append(parts)

df = pd.DataFrame(records, columns=['Line', 'True_Label', 'Log_Entry'])
df['Line']       = df['Line'].astype(int)
df['True_Label'] = df['True_Label'].astype(int)
TOTAL_ATTACKS = int(df['True_Label'].sum())
print(f'Log loaded: {len(df):,} entries | {TOTAL_ATTACKS} attacks | {len(df)-TOTAL_ATTACKS:,} benign')

t0 = time.perf_counter()
df['RawURL'] = df['Log_Entry'].apply(
    lambda x: (METHOD_RE.search(str(x)) or [None, None, None])[2] or '')
df['RawURL'] = df['RawURL'].apply(
    lambda x: re.sub(r'utm_[a-z]+=[^&]*', '',
                     urllib.parse.unquote(str(x)),
                     flags=re.IGNORECASE).strip())
df['QueryValues'] = df['RawURL'].apply(extract_query_values)
parse_ms = (time.perf_counter() - t0) * 1000

df_eval   = df[df['QueryValues'].notna()].copy().reset_index(drop=True)
no_query  = df[df['QueryValues'].isna()]

print(f'\nWay 3 filtering:')
print(f'  Total             : {len(df):,}')
print(f'  No query string   : {len(no_query):,}  (attacks in skipped: {int(no_query["True_Label"].sum())})')
print(f'  To evaluate       : {len(df_eval):,}  (attacks: {int(df_eval["True_Label"].sum())})')
print(f'  Parse time        : {parse_ms:.0f}ms')
print()
print('Sample query values:')
for _, row in df_eval[df_eval['True_Label'] == 0].head(4).iterrows():
    print(f'  [benign] {repr(row["QueryValues"][:60])}')
for _, row in df_eval[df_eval['True_Label'] == 1].head(3).iterrows():
    print(f'  [attack] {repr(row["QueryValues"][:60])}')


Log loaded: 702,389 entries | 28 attacks | 702,361 benign

Way 3 filtering:
  Total             : 702,389
  No query string   : 457,937  (attacks in skipped: 0)
  To evaluate       : 244,452  (attacks: 28)
  Parse time        : 12735ms

Sample query values:
  [benign] '106'
  [benign] '291'
  [benign] '164'
  [benign] '264'
  [attack] '(SELECT 7505 FROM(SELECT COUNT(*),CONCAT(0x7171787671,(SELEC'
  [attack] '(SELECT CONCAT(0x7171787671,(SELECT (ELT(1399=1399,1))),0x71'
  [attack] '1 UNION ALL SELECT CONCAT(0x7171787671,0x537653544175467a724'


## 3. Vectorize with NB03 Vocabulary

In [3]:
vectorizer  = joblib.load('results/models/03_vectorizer.pkl')
queries     = df_eval['QueryValues'].tolist()
y_true      = df_eval['True_Label'].values
BENIGN_SIZE = int((y_true == 0).sum())

t0 = time.perf_counter()
X_eval = hstack([vectorizer.transform(queries), build_symbol_matrix(queries)])
feat_ms = (time.perf_counter() - t0) * 1000

print(f'Vocab (NB03)   : {len(vectorizer.vocabulary_):,}  features: {X_eval.shape[1]:,}')
print(f'Eval samples   : {X_eval.shape[0]:,}  (feat time: {feat_ms:.0f}ms)')


Vocab (NB03)   : 15,185  features: 15,202
Eval samples   : 244,452  (feat time: 9950ms)


## 4. Run All Models — Default Threshold + E2E Latency

In [4]:
model_files = {
    'Logistic Regression': 'results/models/03_logistic_regression_model.pkl',
    'SGD (log loss)':      'results/models/03_sgd_log_loss_model.pkl',
    'LinearSVC':           'results/models/03_linearsvc_model.pkl',
    'Decision Tree':       'results/models/03_decision_tree_model.pkl',
    'Naive Bayes':         'results/models/03_naive_bayes_model.pkl',
    'Random Forest':       'results/models/03_random_forest_model.pkl',
}

sample_qv = [queries[0]]    # single request for E2E timing
results, all_preds, all_probs = [], {}, {}

for name, path in model_files.items():
    model  = joblib.load(path)
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    # Single-request E2E latency
    t0 = time.perf_counter()
    Xs = hstack([vectorizer.transform(sample_qv), build_symbol_matrix(sample_qv)])
    model.predict(Xs)
    e2e_ms = (time.perf_counter() - t0) * 1000

    tp      = int(((y_pred==1) & (y_true==1)).sum())
    fp      = int(((y_pred==1) & (y_true==0)).sum())
    prec    = precision_score(y_true, y_pred, zero_division=0)
    rec_all = tp / TOTAL_ATTACKS if TOTAL_ATTACKS > 0 else 0
    fp10k   = fp / BENIGN_SIZE * 10000

    results.append({
        'Model':       name,
        'TP':          tp,
        'FP':          fp,
        'Recall_all':  round(rec_all, 4),
        'Precision':   round(prec,    4),
        'FP_per_10k':  round(fp10k,   2),
        'E2E_ms':      round(e2e_ms,  3),
    })
    all_preds[name] = y_pred
    all_probs[name] = y_prob
    print(f'{name}: TP={tp}  FP={fp:,}  FP/10k={fp10k:.2f}  Rec={rec_all:.4f}  E2E={e2e_ms:.3f}ms')

results_df = pd.DataFrame(results)
results_df.to_csv('results/metrics/04_model_results_way3.csv', index=False)
print()
print('=== WAY 3 RESULTS (threshold 0.5) ===')
print(results_df[['Model', 'TP', 'FP', 'Precision', 'Recall_all', 'FP_per_10k', 'E2E_ms']].to_string(index=False))


Logistic Regression: TP=28  FP=19,985  FP/10k=817.64  Rec=1.0000  E2E=1.509ms
SGD (log loss): TP=28  FP=18,885  FP/10k=772.63  Rec=1.0000  E2E=2.719ms
LinearSVC: TP=27  FP=20,015  FP/10k=818.86  Rec=0.9643  E2E=7.899ms
Decision Tree: TP=28  FP=21,893  FP/10k=895.70  Rec=1.0000  E2E=2.926ms
Naive Bayes: TP=27  FP=231,827  FP/10k=9484.63  Rec=0.9643  E2E=3.396ms
Random Forest: TP=28  FP=18,755  FP/10k=767.31  Rec=1.0000  E2E=64.147ms

=== WAY 3 RESULTS (threshold 0.5) ===
              Model  TP     FP  Precision  Recall_all  FP_per_10k  E2E_ms
Logistic Regression  28  19985     0.0014      1.0000      817.64   1.509
     SGD (log loss)  28  18885     0.0015      1.0000      772.63   2.719
          LinearSVC  27  20015     0.0013      0.9643      818.86   7.899
      Decision Tree  28  21893     0.0013      1.0000      895.70   2.926
        Naive Bayes  27 231827     0.0001      0.9643     9484.63   3.396
      Random Forest  28  18755     0.0015      1.0000      767.31  64.147


## 5. RF at T_high and T_low

In [5]:
th      = json.load(open('results/models/03_thresholds.json'))
rf_prob = all_probs['Random Forest']
print(f'T_high={th["t_high"]}  T_low={th["t_low"]}'
      f'  (T_high > T_low: {th["t_high"] > th["t_low"]}  ✅)')
print()

for lbl, tv in [('T_high', th['t_high']), ('T_low', th['t_low'])]:
    yp    = (rf_prob >= tv).astype(int)
    tp_   = int(((yp==1) & (y_true==1)).sum())
    fp_   = int(((yp==1) & (y_true==0)).sum())
    prec_ = tp_ / (tp_ + fp_) if tp_ + fp_ > 0 else 0
    rec_  = tp_ / TOTAL_ATTACKS
    print(f'RF at {lbl}={tv}: TP={tp_}  FP={fp_:,}'
          f'  Prec={prec_:.4f}  Rec={rec_:.4f}  FP/10k={fp_/BENIGN_SIZE*10000:.2f}')


T_high=1.0  T_low=0.99  (T_high > T_low: True  ✅)

RF at T_high=1.0: TP=7  FP=6  Prec=0.5385  Rec=0.2500  FP/10k=0.25
RF at T_low=0.99: TP=11  FP=8  Prec=0.5789  Rec=0.3929  FP/10k=0.33


## 6. NB02 Baseline vs NB04 — Direct Comparison

In [6]:
nb02 = pd.DataFrame([
    {'Model': 'Logistic Regression', 'NB02_TP': 19, 'NB02_FP':   5699, 'NB02_FP10k':  81.6},
    {'Model': 'Decision Tree',       'NB02_TP': 23, 'NB02_FP': 112021, 'NB02_FP10k':1603.0},
    {'Model': 'Naive Bayes',         'NB02_TP':  7, 'NB02_FP':    106, 'NB02_FP10k':   1.5},
    {'Model': 'Random Forest',       'NB02_TP': 21, 'NB02_FP':   5666, 'NB02_FP10k':  81.1},
])

nb04_sub = (results_df[['Model', 'TP', 'FP', 'FP_per_10k']]
            .rename(columns={'TP': 'NB04_TP', 'FP': 'NB04_FP', 'FP_per_10k': 'NB04_FP10k'}))
comp = nb02.merge(nb04_sub, on='Model', how='inner')
comp['FP_change_pct'] = ((comp['NB04_FP'] - comp['NB02_FP']) / comp['NB02_FP'] * 100).round(1)

print('=== NB02 BASELINE vs NB04 WAY 3 ===')
print(comp[['Model', 'NB02_TP', 'NB04_TP', 'NB02_FP', 'NB04_FP',
            'NB02_FP10k', 'NB04_FP10k', 'FP_change_pct']].to_string(index=False))
comp.to_csv('results/metrics/04_nb02_vs_nb04_comparison.csv', index=False)

# Bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(comp)); w = 0.35
ax1.bar(x - w/2, comp['NB02_FP'],    w, label='NB02 Baseline', color='tomato')
ax1.bar(x + w/2, comp['NB04_FP'],    w, label='NB04 Way 3',    color='steelblue')
ax1.set_xticks(x); ax1.set_xticklabels(comp['Model'], rotation=15, fontsize=8)
ax1.set_ylabel('False Positives'); ax1.set_title('FP Count: Baseline vs Way 3')
ax1.legend(); ax1.grid(axis='y', alpha=0.4)

ax2.bar(x - w/2, comp['NB02_FP10k'], w, label='NB02 Baseline', color='tomato')
ax2.bar(x + w/2, comp['NB04_FP10k'], w, label='NB04 Way 3',    color='steelblue')
ax2.set_xticks(x); ax2.set_xticklabels(comp['Model'], rotation=15, fontsize=8)
ax2.set_ylabel('FP per 10k'); ax2.set_title('FP Rate: Baseline vs Way 3')
ax2.legend(); ax2.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('results/figures/04_nb02_vs_nb04_comparison.png', dpi=150)
plt.close()
print('Saved: 04_nb02_vs_nb04_comparison.png')


=== NB02 BASELINE vs NB04 WAY 3 ===
              Model  NB02_TP  NB04_TP  NB02_FP  NB04_FP  NB02_FP10k  NB04_FP10k  FP_change_pct
Logistic Regression       19       28     5699    19985        81.6      817.64          250.7
      Decision Tree       23       28   112021    21893      1603.0      895.70          -80.5
        Naive Bayes        7       27      106   231827         1.5     9484.63       218604.7
      Random Forest       21       28     5666    18755        81.1      767.31          231.0
Saved: 04_nb02_vs_nb04_comparison.png


## 7. Summary

In [7]:
th = json.load(open('results/models/03_thresholds.json'))
rf_p = all_probs['Random Forest']
print('=' * 65)
print('NOTEBOOK 04 — COMPLETE (Way 3 Re-evaluation)')
print('=' * 65)
print()
print('WAY 3 RESULTS (threshold 0.5):')
print(results_df[['Model', 'TP', 'FP', 'Recall_all', 'FP_per_10k', 'E2E_ms']].to_string(index=False))
print()
for lbl, tv in [('T_high', th['t_high']), ('T_low', th['t_low'])]:
    tp_ = int(((rf_p >= tv) & (y_true==1)).sum())
    fp_ = int(((rf_p >= tv) & (y_true==0)).sum())
    print(f'RF at {lbl}={tv}: TP={tp_}  FP={fp_:,}'
          f'  FP/10k={fp_/BENIGN_SIZE*10000:.2f}  Rec={tp_/TOTAL_ATTACKS:.4f}')
print()
print('NEXT: Notebook 05 — 3-Tier Decision Policy & Operational Dashboard')


NOTEBOOK 04 — COMPLETE (Way 3 Re-evaluation)

WAY 3 RESULTS (threshold 0.5):
              Model  TP     FP  Recall_all  FP_per_10k  E2E_ms
Logistic Regression  28  19985      1.0000      817.64   1.509
     SGD (log loss)  28  18885      1.0000      772.63   2.719
          LinearSVC  27  20015      0.9643      818.86   7.899
      Decision Tree  28  21893      1.0000      895.70   2.926
        Naive Bayes  27 231827      0.9643     9484.63   3.396
      Random Forest  28  18755      1.0000      767.31  64.147

RF at T_high=1.0: TP=7  FP=6  FP/10k=0.25  Rec=0.2500
RF at T_low=0.99: TP=11  FP=8  FP/10k=0.33  Rec=0.3929

NEXT: Notebook 05 — 3-Tier Decision Policy & Operational Dashboard
